# 4단계 · 공식 Origin 종합 평가와 ronbun 내보내기

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "carbon_mfn1024_s8972_v2"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
PROTOCOL_SCOPE = "paper_table1"
DATASETS = ["tinyface", "scface", "survface"]
HEADS = ["cosface", "adaface", "magface", "arcface"]
EXPORT_BACKBONE = False
REPORT_DIR = None  # 조회할 완료 보고서 폴더를 명시할 때 사용
run = run_directory(MODEL_ROOT, RUN_ID)
validate_choices(DATASETS, HEADS, PROTOCOL_SCOPE, "clean_parent")
print(run / "origin_matrix" / PROTOCOL_SCOPE)

In [ ]:
# cell 2 : 실행
if EXECUTE:
    report = origin_matrix(DATA_ROOT, run, PROTOCOL_SCOPE, DATASETS, HEADS,
                           seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, export=EXPORT_BACKBONE)

In [ ]:
# cell 3 : 결과 확인
if not EXECUTE and REPORT_DIR is not None:
    report = read_result(Path(REPORT_DIR))
if EXECUTE or REPORT_DIR is not None:
    display(pd.DataFrame([dict(head=entry["head"], **row)
                          for entry in report["records"] for row in entry.get("comparison", [])]))
    display(pd.DataFrame([dict(dataset=entry["dataset"], head=entry["head"], status=entry["status"])
                          for entry in report["records"]]))
    survface = []
    for entry in report["records"]:
        if entry["dataset"] == "survface" and entry["status"] == "completed":
            metrics = entry["metrics"]
            survface.extend(dict(head=entry["head"], unknown_cos_ge_095=metrics["unknown_cos_ge_095"], **row)
                            for key, row in metrics.items() if key != "unknown_cos_ge_095")
    if survface:
        display(pd.DataFrame(survface))
    report_dir = Path(report.get("report_dir", REPORT_DIR))
    print("보고서", report_dir)
    if (report_dir / "inventory.json").is_file():
        print("ronbun 인벤토리", report_dir / "inventory.json")
else:
    display(pd.DataFrame([dict(dataset=ds, head=head, receipt=(model_dir(
        run, PROTOCOL_SCOPE, "clean_parent", ds, head) / "COMPLETED").is_file())
        for ds in DATASETS for head in HEADS]))